In [ ]:
import json
import os

feedbacks_tutor = '../5_experiments/tutor_feedbacks.json'

resp_person = []

with open(feedbacks_tutor, 'r', encoding='utf-8') as f:
    data = json.load(f)

    analisys = []
    analise_confusion_matrix = []
    analise_questions = []
    
    for index, feedback_person in enumerate(data):
        # Imprima os valores do feedback
        #print(f"Feedback da pessoa {index + 1}: {feedback_person['feedback']}")
        real_person = []
        predicted_person = []
        
        #Obter o valor do gabarito de cada questão.
        questoes = feedback_person['questions']
        #print("questoes: ", questoes)

        counters = {
            "person_id": 0,
            "true_positive": 0,
            "true_negative": 0,
            "false_positive": 0,
            "false_negative": 0,
            "quantidade_questoes": len(questoes)
        }

        
        for index, questao in enumerate(questoes):
            # print("#######################################################")
            # print("gabarito: ", questao['TX_GABARITO'])
            # print("resposta: ", feedback_person['resp_person'][index]['resposta'])
            # print("feedback: ", feedback_person['feedback'][index]['resposta'])
           
            acerto_person_resp = 0
            acerto_tutor_resp = 0
            
            if questao['TX_GABARITO'] == feedback_person['resp_person'][index]['resposta']:
                acerto_person_resp = 1
            else:
                acerto_person_resp = 0

            #VP (True Positive) -> modelo acertou o positivo. O aluno acertou e o LLM disse que estava correto.
            if questao['TX_GABARITO'] == feedback_person['resp_person'][index]['resposta'] and feedback_person['feedback'][index]['resposta'] == "C":
                #print(f"Questão {questao['CO_POSICAO']} - Resposta correta")
                counters["true_positive"] += 1
                acerto_tutor_resp = 1
            
            #VN (True Negative) -> modelo acertou o negativo. O aluno errou e o LLM disse que estava incorreto.
            if questao['TX_GABARITO'] != feedback_person['resp_person'][index]['resposta'] and feedback_person['feedback'][index]['resposta'] == "E":
                #print(f"Questão {questao['CO_POSICAO']} - Resposta incorreta")
                counters["true_negative"] += 1
                acerto_tutor_resp = 1
           
            #FP (False Positive) -> modelo errou dizendo “positivo” quando era “negativo”. O aluno errou e o LLM disse que estava correto.
            if questao['TX_GABARITO'] != feedback_person['resp_person'][index]['resposta'] and feedback_person['feedback'][index]['resposta'] == "C":
                #print(f"Questão {questao['CO_POSICAO']} - Resposta incorreta")
                counters["false_positive"] += 1
                acerto_tutor_resp = 0

            #FN (False Negative) -> modelo errou dizendo “negativo” quando era “positivo”. O aluno acerta e o LLM disse que estava incorreto.
            if questao['TX_GABARITO'] == feedback_person['resp_person'][index]['resposta'] and feedback_person['feedback'][index]['resposta'] == "E":
                #print(f"Questão {questao['CO_POSICAO']} - Resposta incorreta")
                counters["false_negative"] += 1
                acerto_tutor_resp = 0

            if acerto_tutor_resp == 1:
                predicted_person.append(1)
            else:
                predicted_person.append(0)
            
            real_person.append(acerto_person_resp)

            analise_question = {
                "person_id": feedback_person['id_person'],
                "ano": questao['ano'],
                "area_descricao": questao['descricao_area'],
                "area_sigla": questao['SG_AREA'],
                "questao_id": questao['CO_POSICAO'],
                "gabarito": questao['TX_GABARITO'],
                "param_a": questao['NU_PARAM_A'],
                "param_b": questao['NU_PARAM_B'],
                "param_c": questao['NU_PARAM_C'],
                "resposta_person": feedback_person['resp_person'][index]['resposta'],
                "resposta_tutor": feedback_person['feedback'][index]['resposta'],
                "acerto_person_resp": acerto_person_resp,
                "acerto_tutor_resp": acerto_tutor_resp
            }

            analise_questions.append(analise_question)

        #VP (True Positive) -> modelo acertou o positivo. O aluno acertou e o LLM disse que estava correto.
        #VN (True Negative) -> modelo acertou o negativo. O aluno errou e o LLM disse que estava incorreto.
        #FP (False Positive) -> modelo errou dizendo “positivo” quando era “negativo”. O aluno errou e o LLM disse que estava correto.
        #FN (False Negative) -> modelo errou dizendo “negativo” quando era “positivo”. O aluno acerta e o LLM disse que estava incorreto.

        analise = {
            "person_id": feedback_person['id_person'],
            "true_positive": counters["true_positive"],
            "true_negative": counters["true_negative"],
            "false_positive": counters["false_positive"],
            "false_negative": counters["false_negative"],
            "quantidade_questoes": len(questoes)
        }

        analise_confusion_matrix.append(analise)
        
        # Criar diretório se não existir
        dataset_dir = './'
        os.makedirs(dataset_dir, exist_ok=True)

        # Salvar o dataset em JSON
        output_path = os.path.join(dataset_dir, 'analise_confusion_matrix.json')
        with open(output_path, 'w', encoding='utf-8') as f:
              json.dump(analise_confusion_matrix, f, ensure_ascii=False, indent=2)

        output_path2 = os.path.join(dataset_dir, 'analise_questions.json')
        with open(output_path2, 'w', encoding='utf-8') as f:
              json.dump(analise_questions, f, ensure_ascii=False, indent=2)


In [ ]:
# O arquivo .json possui a seguinte estrutura:
#  {
#     "id_person": 1,
#     "questions": [
#       {
#         "enunciado": "\nQUESTÃO 22\t\nTEXTO I\nCapítulo 4, versículo 3\nMinha palavra vale um tiro, eu tenho muita munição\nNa queda ou na ascensão, minha atitude vai além\nE tem disposição pro mal e pro bem\nTalvez eu seja um sádico ou um anjo, um mágico\nOu juiz, ou réu, o bandido do céu\nMalandro ou otário, quase sanguinário\nFranco atirador se for necessário\nRevolucionário, insano, ou marginal\nAntigo e moderno, imortal\nFronteira do céu com o inferno\nAstral imprevisível, como um ataque cardíaco do verso.\nRACIONAIS MCs. Sobrevivendo ao inferno.  \nSão Paulo: Cosa Nostra, 1997 (fragmento).\nTEXTO II\nPode-se dizer que as várias experiências narradas \nnos discos do Racionais tratam no fundo de um só tema: \na violência que estrutura a nossa sociedade. O grupo \ncanta a violência que estrutura as relações entre os \nfamiliares, os amigos, o homem e a mulher, o traficante e \no viciado. Canta a violência do crime. A violência causada \npor inveja ou por vaidade. Também canta que a relação \nentre as classes sociais é sempre violenta: o racismo, \na miséria, os baixos salários, a concentração de renda,  \na esmola, a publicidade, o alcoolismo, o jornalismo, o poder \npolicial, a justiça, o sistema penitenciário, o governo existem \npor meio da violência.\nGARCIA, W. Ouvindo Racionais MCs. Teresa: revista de  \nliteratura brasileira, n. 5, 2004 (adaptado).\nNa letra da canção, a tematização da violência mencionada \nno Texto II manifesta-se\nA\tcomo metáfora da desigualdade, que associa a ideia de \njustiça a valores históricos negativos.\nB\tna referência a termos bélicos, que sinaliza uma crítica \nsocial à opressão da população das periferias.\nC\tcomo procedimento metalinguístico, que concebe a \npalavra como uma forma de combate e insubordinação.\nD\tnas definições ambíguas do enunciador, que inverte e \nrelativiza as representações da maldade e da bondade.\nE\tna menção à imortalidade, que sugere a possibilidade de \nresistência para além da dicotomia entre vida e morte.\n\nENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM20E4ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024\nENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM20E4ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024\nENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024ENEM2024\nLINGUAGENS, CÓDIGOS E SUAS TECNOLOGIAS E REDAÇÃO • 1º DIA • CADERNO 1 • AZUL •\n10",
#         "ano": 2024,
#         "descricao_area": "linguagens, códigos e suas tecnologias",
#         "SG_AREA": "LC",
#         "CO_POSICAO": 22.0,
#         "TP_LINGUA": null,
#         "TX_GABARITO": "C",
#         "CO_HABILIDADE": 22.0,
#         "NU_PARAM_A": 2.2483400000000002,
#         "NU_PARAM_B": 1.8179800000000002,
#         "NU_PARAM_C": 0.17832
#       }
#     ],
#     "resp_person": [
#       {
#         "ano": 2024,
#         "CO_POSICAO": 22.0,
#         "descricao_area": "linguagens, códigos e suas tecnologias",
#         "resposta": "D"
#       }
#     ],
#     "feedback": [
#       {
#         "numero_questao": "22",
#         "resposta": "C",
#         "feedback_tutor": "Parabéns, você acertou! A letra realmente apresenta definições ambíguas que invertem e relativizam as representações de maldade e bondade, evidenciando a violência como elemento central nas relações sociais, conforme destacado no Texto II. Continue estudando as estratégias de linguagem nas músicas e textos críticos para aprofundar sua compreensão.###"
#       }
#     ]
#   },